# Lesson 3A: Monte Carlo Methods - Theory

## Introduction

In Lessons 1-2, we learned **Dynamic Programming (DP)**: algorithms that solve MDPs when we **know** the environment dynamics (P, R).

Now we learn **Monte Carlo (MC) methods**: learning when we **don't know** the dynamics, only episodes of experience.

### Key Transition

**DP**: Given MDP model → compute V* and π*

**MC**: Given episodes of interaction → learn V* and π*

This is the **model-free** learning setting. Instead of exact transitions P(s'|s,a), we sample trajectories.

### Where MC Fits

**Advantages**:
- No need to know environment model
- Learn from real or simulated experience
- Can focus on relevant parts of state space
- Off-policy learning possible (learn optimal policy from suboptimal experience)

**Challenges**:
- Requires full episodes (can't learn from incomplete trajectories)
- Higher variance than DP (learning from samples)
- Slower convergence

### Learning Objectives

1. Understand episodic vs. continuing task distinction
2. Learn **Monte Carlo Prediction**: estimate V^π from episodes
3. Learn **Monte Carlo Control**: find optimal π iteratively
4. Distinguish **on-policy** vs **off-policy** learning
5. Master **importance sampling** for off-policy learning

## Table of Contents

1. [Setup](#setup)
2. [Episodic vs. Continuing Tasks](#episodic)
3. [Monte Carlo Prediction](#prediction)
4. [First-Visit vs. Every-Visit](#fv-ev)
5. [Monte Carlo Control](#control)
6. [Exploring Starts](#exploring-starts)
7. [Epsilon-Greedy Exploration](#epsilon-greedy)
8. [Off-Policy Learning](#off-policy)
9. [Importance Sampling](#importance-sampling)
10. [Canonical Example: Blackjack](#blackjack)
11. [Comparison with DP](#comparison)
12. [Summary](#summary)

<a name="setup"></a>
## Setup

In [ ]:
import subprocess
import sys

packages = ['numpy', 'matplotlib', 'seaborn', 'pandas', 'scipy', 'gymnasium']
for pkg in packages:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from collections import defaultdict
from typing import Dict, List, Tuple, Any
import gymnasium as gym

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print('✅ All libraries loaded!')

<a name="episodic"></a>
## Episodic vs. Continuing Tasks

### Episodic Tasks

**Episodes**: Finite sequences of states and actions that end at a terminal state.

Example: Game of chess
- Start: Initial position
- Actions: Moves
- End: Checkmate or draw (terminal state)
- Return: +1 (win), 0 (draw), -1 (loss)

Each episode is independent. Learning treats each episode separately.

**Trajectory notation**:
$$S_0, A_0, R_1, S_1, A_1, R_2, ..., S_T (\text{terminal})$$

where T is the terminal time step (varies per episode).

### Continuing Tasks

**No terminal state**: Process continues indefinitely.

Example: Robot control
- No natural endpoint
- Discount factor γ < 1 makes infinite returns finite
- DP methods handle naturally
- MC methods need special handling

### For This Lesson

We focus on **episodic tasks** because MC methods are naturally suited to them. Continuing tasks are handled similarly with modifications (return truncation, etc.).

In [ ]:
# Visualize episode structure
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Episodic
ax = axes[0]
ax.text(0.5, 0.95, 'EPISODIC TASK', ha='center', fontsize=13, fontweight='bold', transform=ax.transAxes)

episode_states = ['S₀', 'S₁', 'S₂', 'S₃', 'Terminal']
x_pos = np.linspace(0.1, 0.9, len(episode_states))

for i, (pos, state) in enumerate(zip(x_pos, episode_states)):
    if state == 'Terminal':
        color = 'lightcoral'
    else:
        color = 'lightblue'
    
    circle = plt.Circle((pos, 0.6), 0.04, transform=ax.transAxes, color=color, ec='black', linewidth=2)
    ax.add_patch(circle)
    ax.text(pos, 0.6, state, ha='center', va='center', fontsize=9, fontweight='bold', transform=ax.transAxes)
    
    if i < len(episode_states) - 1:
        ax.annotate('', xy=(x_pos[i+1]-0.045, 0.6), xytext=(pos+0.045, 0.6),
                   arrowprops=dict(arrowstyle='->', lw=2), xycoords=ax.transAxes)
        ax.text((pos+x_pos[i+1])/2, 0.75, f'A_{i}, R_{i+1}', ha='center', fontsize=9, transform=ax.transAxes)

ax.text(0.5, 0.35, 'Fixed number of steps per episode\nEach episode is independent', 
       ha='center', fontsize=10, transform=ax.transAxes, style='italic')
ax.text(0.5, 0.15, 'G_t = R_{t+1} + R_{t+2} + ... + R_T\n(sum to terminal)', 
       ha='center', fontsize=10, family='monospace', transform=ax.transAxes,
       bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

# Continuing
ax = axes[1]
ax.text(0.5, 0.95, 'CONTINUING TASK', ha='center', fontsize=13, fontweight='bold', transform=ax.transAxes)

states = ['S₀', 'S₁', 'S₂', '...', 'S_∞']
x_pos_cont = np.linspace(0.1, 0.9, len(states))

for i, (pos, state) in enumerate(zip(x_pos_cont, states)):
    if state == 'S_∞':
        color = 'lightyellow'
    else:
        color = 'lightblue'
    
    if state != '...':
        circle = plt.Circle((pos, 0.6), 0.04, transform=ax.transAxes, color=color, ec='black', linewidth=2)
        ax.add_patch(circle)
    
    ax.text(pos, 0.6, state, ha='center', va='center', fontsize=9, fontweight='bold', transform=ax.transAxes)
    
    if i < len(states) - 1 and state != '...':
        ax.annotate('', xy=(x_pos_cont[i+1]-0.045, 0.6), xytext=(pos+0.045, 0.6),
                   arrowprops=dict(arrowstyle='->', lw=2), xycoords=ax.transAxes)

ax.text(0.5, 0.35, 'No terminal state; process continues forever\nNeed γ < 1 or truncation', 
       ha='center', fontsize=10, transform=ax.transAxes, style='italic')
ax.text(0.5, 0.15, 'G_t = R_{t+1} + γR_{t+2} + γ²R_{t+3} + ...\n(infinite sum, γ makes it finite)', 
       ha='center', fontsize=10, family='monospace', transform=ax.transAxes,
       bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

plt.tight_layout()
plt.show()

<a name="prediction"></a>
## Monte Carlo Prediction

### The Idea

Given a policy π and episodes generated by following π:

$$V^\pi(s) \approx \text{Average return from state } s$$

**Key insight**: Estimate value by averaging observed returns.

### Algorithm

```
Input: policy π
Initialize: Returns(s) ← empty list for all s

loop:
  Generate episode: S_0, A_0, R_1, S_1, A_1, R_2, ..., S_T
  G ← 0
  for t = T-1 down to 0:
    G ← R_{t+1} + γG
    Append G to Returns(S_t)

V^π(s) ← average(Returns(s)) for all s
```

### Key Properties

1. **Unbiased**: E[V(s)] = V^π(s) (converges to true value)
2. **Independent of model**: Only needs episodes
3. **Converges slowly**: O(1/n) variance (vs DP which is exact)
4. **Episodic**: Requires complete episodes

<a name="fv-ev"></a>
## First-Visit vs. Every-Visit MC

### First-Visit MC

Only count the **first occurrence** of state s in each episode.

Example: Episode S₀, S₁, S₂, S₁, S₃
- Visit to S₁ at t=1: Include return
- Visit to S₁ at t=3: **Skip** (not first visit)

### Every-Visit MC

Count **all occurrences** of state s in each episode.

Example: Same episode S₀, S₁, S₂, S₁, S₃
- Visit to S₁ at t=1: Include return
- Visit to S₁ at t=3: **Include** return (different return!)

### Comparison

| Property | First-Visit | Every-Visit |
|----------|-------------|-------------|
| Returns used | Fewer (first only) | More (all) |
| Variance | Higher | Lower |
| Bias | Unbiased | Unbiased |
| Convergence | Slower | Faster |
| Preference | Historical | Modern |

**Modern practice**: Every-visit is faster without sacrificing properties.

In [ ]:
# Example: Difference between first-visit and every-visit
print('Example Episode: S0 → S1 → S2 → S1 → S3 → Terminal')
print('Rewards:        r=0,  r=1,  r=2,  r=-1, r=10')
print('\nReturns (γ=1 for simplicity):')
print('From S0: 0 + 1 + 2 - 1 + 10 = 12')
print('From S1 (t=1): 1 + 2 - 1 + 10 = 12')
print('From S2: 2 - 1 + 10 = 11')
print('From S1 (t=3): -1 + 10 = 9')
print('From S3: 10')
print()
print('FIRST-VISIT MC:')
print('  Returns(S0): [12]           → V(S0) = 12')
print('  Returns(S1): [12]           → V(S1) = 12  (only first visit at t=1)')
print('  Returns(S2): [11]           → V(S2) = 11')
print('  Returns(S3): [10]           → V(S3) = 10')
print()
print('EVERY-VISIT MC:')
print('  Returns(S0): [12]           → V(S0) = 12')
print('  Returns(S1): [12, 9]        → V(S1) = 10.5  (both visits!)')
print('  Returns(S2): [11]           → V(S2) = 11')
print('  Returns(S3): [10]           → V(S3) = 10')
print()
print('Key: Every-visit sees S1 twice with different returns')

<a name="control"></a>
## Monte Carlo Control

### Goal

Find optimal policy π* by learning from episodes.

### GPI Structure

Like DP, MC uses Generalized Policy Iteration:

```
loop:
  EVALUATION: Estimate Q^π from episodes (MC Prediction)
  IMPROVEMENT: Improve policy: π(s) ← argmax_a Q(s, a)
```

Key difference: Estimate **Q(s,a)** values instead of **V(s)**.

Why? For improvement we need to know value of each **action**, not just states.

### Algorithm: MC Control with Exploring Starts

```
Initialize: π(s) ← arbitrary policy
Initialize: Returns(s,a) ← empty list for all s, a

loop:
  Generate episode with exploring starts (see next section)
  for each (s,a) in episode:
    Append return to Returns(s,a)
  
  for each s:
    Q(s, a) ← average(Returns(s,a)) for each a visited in episode
    π(s) ← argmax_a Q(s, a)
```

### Convergence

Converges to π* in the limit (with conditions on exploration).

<a name="exploring-starts"></a>
## Exploring Starts

### The Exploration Challenge

For optimal policy improvement, we need to estimate Q(s,a) for **all** state-action pairs.

But if policy is greedy (π(s) = argmax_a Q(s,a)), we might not visit all actions:
- Only actions chosen by π are tried
- Never discover if an untried action is better
- Policy improvement stalls

### Solution: Exploring Starts

Guarantee that all state-action pairs are visited infinitely often by starting episodes with random (s, a) pairs:

```
while generating episode:
  Randomly pick (S0, A0)
  Then follow policy π: S0 -A0-> R1, S1 -π(S1)-> ...
```

**Result**: Every state-action pair is first state in some episode → visited infinitely often.

### Drawback

Exploring starts assume we can force arbitrary (s,a) pairs, which is unrealistic for real applications.

Solution: Epsilon-greedy exploration (next section).

<a name="epsilon-greedy"></a>
## Epsilon-Greedy Exploration

### Motivation

In real environments, we can't force arbitrary starting states. Instead, we need to explore during normal play.

### Algorithm

At each state, with probability ε:
- Pick a **random action** (exploration)

With probability 1-ε:
- Pick the **greedy action** argmax_a Q(s,a) (exploitation)

```python
if random() < epsilon:
    action = random_action()  # Explore
else:
    action = argmax_a(Q[state])  # Exploit
```

### Convergence

**Theorem**: If ε > 0 (explore infinitely often) and learning rates decay appropriately, MC with ε-greedy converges to ε-optimal policy:

$$V^{\pi_\epsilon}(s) \geq V^*(s) - \frac{2\epsilon}{1-\gamma} \max_a r(a)$$

Can make ε → 0 over time (annealing) to converge to π*.

### Trade-off

- ε too high: Converges to suboptimal policy
- ε too low: Explores too little, slow to find good policy
- Sweet spot: Often ε = 0.1 works well empirically

<a name="off-policy"></a>
## Off-Policy Learning

### On-Policy vs. Off-Policy

**On-Policy**: Learn the policy you're following.
- MC control with ε-greedy learns π_ε (suboptimal)
- Data comes from executing π, learn the same π

**Off-Policy**: Learn π* while following different behavior policy μ.
- Execute suboptimal or exploratory behavior policy μ
- Learn optimal policy π* from this experience
- More efficient: reuse old data with new policies

### Advantage

Can learn from:
- Suboptimal policies (learn to improve them)
- Human demonstrations
- Old policy logs
- Multiple policies simultaneously

### Challenge

Behavior μ(a|s) ≠ Target π(a|s) leads to different action distributions.

Naive averaging gives biased estimates. Solution: **Importance sampling**.

<a name="importance-sampling"></a>
## Importance Sampling

### Problem Setup

We have returns generated by behavior policy μ:
- Trajectory: S₀, A₀, R₁, S₁, ..., collected under μ
- Return: G = R₁ + γR₂ + ... + γ^(T-1)R_T

We want to estimate V^π(s) (value under target policy π) using data from μ.

### Importance Sampling Ratio

Probability of trajectory under π vs under μ:

$$\rho_{t:T} = \frac{\prod_{k=t}^{T-1} \pi(A_k | S_k) P(S_{k+1} | S_k, A_k)}{\prod_{k=t}^{T-1} \mu(A_k | S_k) P(S_{k+1} | S_k, A_k)} = \frac{\prod_{k=t}^{T-1} \pi(A_k | S_k)}{\prod_{k=t}^{T-1} \mu(A_k | S_k)}$$

**Key**: Environment dynamics P cancel! Only policies matter.

### Weighted Average

Unbiased estimate of V^π(s):

$$V^\pi(s) \approx \frac{\sum \rho_{t:T} G_t}{\sum \rho_{t:T}}$$

### Problem: High Variance

If π and μ are very different, importance ratios become huge/tiny → extremely high variance.

Solution: **Weighted importance sampling** (slightly biased but lower variance).

<a name="blackjack"></a>
## Canonical Example: Blackjack

### Game Rules

Goal: Get hand value as close to 21 as possible without going over.

**States**: (player_sum, dealer_showing_card, player_has_ace)
- player_sum: 12-21 (usable aces give 10+ points)
- dealer card: 2-11
- has_usable_ace: true/false
- ~200 states

**Actions**: Hit (draw card) or Stand (stop)

**Reward**: +1 (win), -1 (lose), 0 (draw)

### Why Ideal for MC Learning

1. **Simple**: Easy to simulate
2. **Stochastic**: Random card draws
3. **No known model**: Can't write P directly (card probabilities complex)
4. **Episodic**: Hand finishes in ~10-20 steps
5. **Visualizable**: 2D state space easy to plot

<a name="comparison"></a>
## Comparison: MC vs. DP

| Aspect | Dynamic Programming | Monte Carlo |
|--------|--------------------|--------------|
| **Model** | Requires P, R | Doesn't need model |
| **Learning** | Exact (from model) | Stochastic (from samples) |
| **Episodes** | Can use bootstrapping | Requires full episodes |
| **Variance** | None (exact) | High (from sampling) |
| **Bias** | Unbiased | Unbiased (if correct) |
| **Convergence** | Fast (exact) | Slow (1/√n) |
| **Exploration** | Not needed (know dynamics) | Must explore (ε-greedy) |
| **Off-policy** | Difficult | Possible (importance sampling) |
| **Continuing tasks** | Natural (γ handles infinity) | Needs truncation or special handling |
| **Best for** | Known models | Unknown models, real experience |

### When to Use

**Use DP**: Simulator or known environment (games, planning)

**Use MC**: Real-world data, no model, episodic tasks

**Use Hybrid (Temporal Difference)**: Best of both (Lesson 4)

<a name="summary"></a>
## Summary

### Key Concepts

1. **Episodic tasks**: Finite trajectories with clear endings
2. **MC Prediction**: Average returns to estimate V^π
3. **First-visit vs. Every-visit**: Different ways to average returns
4. **MC Control**: Learn π* via policy iteration on Q-values
5. **Exploring Starts**: Force all state-action pairs at start
6. **Epsilon-Greedy**: Explore randomly during normal play
7. **On-Policy vs. Off-Policy**: Learn same or different policy
8. **Importance Sampling**: Weight off-policy returns by probability ratio

### Algorithm Summary

**MC Prediction** (estimate V^π):
- Generate episodes under π
- Average returns to each state
- Converge to V^π

**MC Control** (find π*):
- Alternate: estimate Q^π from episodes
- Improve: π ← greedy(Q)
- Explore enough to try all actions

### Next: Lesson 3B (Practical)

Implement MC on Blackjack, analyze convergence, compare policies.